In [1]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Cấu hình đường dẫn dữ liệu
DATA_DIR = "/Users/nguyenminhtri/FinalYearPro/data/raw"
MAIN_TRAIN_FILE = os.path.join(DATA_DIR, "application_train.csv")

# Nạp dữ liệu thô
df_train = pd.read_csv(MAIN_TRAIN_FILE)
print(f"✅ Đã nạp thành công dữ liệu thô: {df_train.shape[0]:,} dòng | {df_train.shape[1]} cột.")

✅ Đã nạp thành công dữ liệu thô: 307,511 dòng | 122 cột.


In [2]:
# 1. Tách nhãn y và loại bỏ cột ID (SK_ID_CURR) để tránh overfitting
y = df_train['TARGET']
X = df_train.drop(columns=['TARGET', 'SK_ID_CURR'])

# 2. Chuyển đổi các cột Dạng chữ (object/category) sang Dạng số bằng LabelEncoder
categorical_cols = X.select_dtypes(include=['object', 'category']).columns

for col in categorical_cols:
    le = LabelEncoder()
    # Chuyển missing value thành chuỗi 'Missing' trước khi encode
    X[col] = le.fit_transform(X[col].astype(str))

print(f"⚙️ Đã mã hóa {len(categorical_cols)} cột dạng chữ sang dạng số.")
print(f"📊 Kích thước ma trận đặc trưng X đưa vào Baseline: {X.shape}")

⚙️ Đã mã hóa 16 cột dạng chữ sang dạng số.
📊 Kích thước ma trận đặc trưng X đưa vào Baseline: (307511, 120)


In [3]:
# Cấu hình 5-Fold Cross Validation (Bảo toàn tỷ lệ nhãn TARGET)
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Mảng lưu kết quả dự đoán Out-of-Fold (OOF)
oof_preds = np.zeros(X.shape[0])

# Cấu hình tham số mặc định (Default) cho LightGBM Baseline
lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1
}

print("🚀 BẮT ĐẦU HUẤN LUYỆN LIGHTGBM BASELINE MODEL...")
print("=" * 55)

for fold, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]

    # Kích hoạt mô hình
    model = lgb.LGBMClassifier(**lgb_params)

    # Huấn luyện mô hình với Early Stopping (dừng nếu 50 vòng không tăng AUC)
    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # Dự đoán xác suất cho tập Validation
    preds = model.predict_proba(X_valid)[:, 1]
    oof_preds[valid_idx] = preds

    # Tính điểm ROC-AUC từng Fold
    fold_auc = roc_auc_score(y_valid, preds)
    print(f"  ➜ Fold {fold + 1} ROC-AUC: {fold_auc:.5f}")

# Tính tổng điểm ROC-AUC Out-Of-Fold (OOF Benchmark Score)
overall_auc = roc_auc_score(y, oof_preds)
print("=" * 55)
print(f"🎯 ĐIỂM BASELINE ROC-AUC BENCHMARK CUỐI CÙNG: {overall_auc:.5f}")

🚀 BẮT ĐẦU HUẤN LUYỆN LIGHTGBM BASELINE MODEL...
  ➜ Fold 1 ROC-AUC: 0.75414
  ➜ Fold 2 ROC-AUC: 0.76422
  ➜ Fold 3 ROC-AUC: 0.75791
  ➜ Fold 4 ROC-AUC: 0.76368
  ➜ Fold 5 ROC-AUC: 0.75380
🎯 ĐIỂM BASELINE ROC-AUC BENCHMARK CUỐI CÙNG: 0.75869


In [5]:
import xgboost as xgb

# 1. Khởi tạo K-Fold và mảng lưu kết quả OOF cho XGBoost
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
xgb_oof_preds = np.zeros(X.shape[0])

# 2. Cấu hình tham số cơ bản cho XGBoost Baseline
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',        # Sử dụng thuật toán Histogram để tăng tốc tối đa trên Mac
    'learning_rate': 0.05,
    'max_depth': 6,
    'random_state': 42,
    'n_jobs': -1
}

print("🚀 BẮT ĐẦU HUẤN LUYỆN XGBOOST BASELINE MODEL...")
print("=" * 55)

for fold, (train_idx, valid_idx) in enumerate(folds.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]

    # Khai báo mô hình XGBoost
    model_xgb = xgb.XGBClassifier(**xgb_params, n_estimators=1000, early_stopping_rounds=50)

    # Huấn luyện mô hình
    model_xgb.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=False
    )

    # Dự đoán xác suất cho tập Validation
    preds = model_xgb.predict_proba(X_valid)[:, 1]
    xgb_oof_preds[valid_idx] = preds

    fold_auc = roc_auc_score(y_valid, preds)
    print(f"  ➜ XGBoost Fold {fold + 1} ROC-AUC: {fold_auc:.5f}")

# Tổng kết điểm Baseline của XGBoost
xgb_overall_auc = roc_auc_score(y, xgb_oof_preds)
print("=" * 55)
print(f"🎯 ĐIỂM XGBOOST BASELINE ROC-AUC: {xgb_overall_auc:.5f}")

🚀 BẮT ĐẦU HUẤN LUYỆN XGBOOST BASELINE MODEL...
  ➜ XGBoost Fold 1 ROC-AUC: 0.75620
  ➜ XGBoost Fold 2 ROC-AUC: 0.76686
  ➜ XGBoost Fold 3 ROC-AUC: 0.75828
  ➜ XGBoost Fold 4 ROC-AUC: 0.76533
  ➜ XGBoost Fold 5 ROC-AUC: 0.75645
🎯 ĐIỂM XGBOOST BASELINE ROC-AUC: 0.76057


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# 1. Điền dữ liệu thiếu (NaN) bằng giá trị Median cho Random Forest (chỉ tốn vài giây)
print("⚙️ Đang xử lý Missing Values cho Random Forest...")
imputer = SimpleImputer(strategy='median')
X_rf = imputer.fit_transform(X)

# 2. Khởi tạo K-Fold và mảng lưu kết quả OOF cho Random Forest
rf_oof_preds = np.zeros(X_rf.shape[0])

print("🚀 BẮT ĐẦU HUẤN LUYỆN RANDOM FOREST BASELINE MODEL...")
print("=" * 55)

for fold, (train_idx, valid_idx) in enumerate(folds.split(X_rf, y)):
    X_train, y_train = X_rf[train_idx], y.iloc[train_idx]
    X_valid, y_valid = X_rf[valid_idx], y.iloc[valid_idx]

    # Sử dụng 100 cây quyết định để Random Forest chạy nhanh gọn
    model_rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=12,             # Khống chế độ sâu để tránh ngốn RAM và Overfitting
        random_state=42,
        n_jobs=-1
    )

    model_rf.fit(X_train, y_train)

    preds = model_rf.predict_proba(X_valid)[:, 1]
    rf_oof_preds[valid_idx] = preds

    fold_auc = roc_auc_score(y_valid, preds)
    print(f"  ➜ Random Forest Fold {fold + 1} ROC-AUC: {fold_auc:.5f}")

# Tổng kết điểm Baseline của Random Forest
rf_overall_auc = roc_auc_score(y, rf_oof_preds)
print("=" * 55)
print(f"🎯 ĐIỂM RANDOM FOREST BASELINE ROC-AUC: {rf_overall_auc:.5f}")

⚙️ Đang xử lý Missing Values cho Random Forest...
🚀 BẮT ĐẦU HUẤN LUYỆN RANDOM FOREST BASELINE MODEL...
  ➜ Random Forest Fold 1 ROC-AUC: 0.73589
  ➜ Random Forest Fold 2 ROC-AUC: 0.74330
  ➜ Random Forest Fold 3 ROC-AUC: 0.73185
  ➜ Random Forest Fold 4 ROC-AUC: 0.73980
  ➜ Random Forest Fold 5 ROC-AUC: 0.73309
🎯 ĐIỂM RANDOM FOREST BASELINE ROC-AUC: 0.73674


### 📌 Baseline Experiment Summary

- **XGBoost Baseline:** ROC-AUC = 0.76057 (Highest raw accuracy)
- **LightGBM Baseline:** ROC-AUC = 0.75869 (Best trade-off between speed and accuracy)
- **Random Forest Baseline:** ROC-AUC = 0.73674 (Bagging baseline reference)

**Conclusion & Next Steps:**
1. The baseline benchmark is established at **ROC-AUC ~ 0.7606**.
2. Both **LightGBM** and **XGBoost** will be selected as the primary algorithms for subsequent feature engineering and hyperparameter tuning phases.
3. Next step: Proceed to feature_engineering

In [1]:
import os
import time
import psutil
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
import warnings
warnings.filterwarnings('ignore')

def get_current_ram_gb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)

def evaluate_baseline_logistic(file_path: str, n_splits: int = 5):
    print("=" * 85)
    print(f"📂 TEST BASELINE LOGISTIC REGRESSION (RAW DATA): {file_path}")
    print("=" * 85)

    if not os.path.exists(file_path):
        print(f"❌ Lỗi: Không tìm thấy file tại '{file_path}'")
        return

    df = pd.read_parquet(file_path) if file_path.endswith('.parquet') else pd.read_csv(file_path)

    n_rows, n_cols = df.shape
    dataset_ram_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

    target_col = 'TARGET'
    ignore_cols = ['SK_ID_CURR', target_col]
    X = df.drop(columns=[col for col in ignore_cols if col in df.columns])
    y = df[target_col]

    # One-Hot Encoding cho các biến phân loại của bảng gốc
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    if len(cat_cols) > 0:
        X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

    # Thay thế các giá trị vô cực (nếu có) và ép kiểu float32 để tiết kiệm RAM
    X = X.replace([np.inf, -np.inf], np.nan).astype(np.float32)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    lr_oof = np.zeros(len(X))

    def calc_ks(y_true, y_pred):
        return ks_2samp(y_pred[y_true == 1], y_pred[y_true == 0]).statistic * 100

    lr_tr_aucs, lr_va_aucs = [], []
    lr_latencies = []

    # Pipeline chuẩn: Điền median -> Chuẩn hóa z-score -> Logistic Regression
    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(
            C=0.01,
            max_iter=1000,
            solver='lbfgs',
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        ))
    ])

    peak_ram = get_current_ram_gb()
    start_time = time.time()

    print(f"⚡ Đang chạy {n_splits}-Fold Cross Validation cho Baseline...")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_va, y_va = X.iloc[val_idx], y.iloc[val_idx]

        pipeline.fit(X_tr, y_tr)
        peak_ram = max(peak_ram, get_current_ram_gb())

        tr_pred = pipeline.predict_proba(X_tr)[:, 1]
        tr_auc = roc_auc_score(y_tr, tr_pred)
        lr_tr_aucs.append(tr_auc)

        t0_inf = time.time()
        va_pred = pipeline.predict_proba(X_va)[:, 1]
        lr_latencies.append(((time.time() - t0_inf) / len(X_va)) * 1000)
        lr_oof[val_idx] = va_pred

        va_auc = roc_auc_score(y_va, va_pred)
        lr_va_aucs.append(va_auc)

        gap = tr_auc - va_auc
        print(f"   ▫️ Fold {fold} | LR Val: {va_auc:.5f} (Gap: {gap:+.5f})")

    total_time = time.time() - start_time
    lr_total_auc = roc_auc_score(y, lr_oof)
    lr_gaps = np.array(lr_tr_aucs) - np.array(lr_va_aucs)

    print("\n" + "=" * 60)
    print("🏆 KẾT QUẢ ROC-AUC BASELINE LOGISTIC REGRESSION")
    print("=" * 60)
    print(f"🔹 Logistic Regression : {lr_total_auc:.5f}")
    print("=" * 60)

    perf_data = [{
        'Model': 'Baseline (Logistic Regression)',
        'ROC-AUC': f"{lr_total_auc:.5f}",
        'Fold AUC (Mean ± Std)': f"{np.mean(lr_va_aucs):.5f} ± {np.std(lr_va_aucs):.5f}",
        'Train AUC': f"{np.mean(lr_tr_aucs):.5f}",
        'Gap (Mean ± Std)': f"{np.mean(lr_gaps):.5f} ± {np.std(lr_gaps):.5f}",
        'Gini (%)': f"{(2 * lr_total_auc - 1) * 100:.2f}%",
        'KS (%)': f"{calc_ks(y, lr_oof):.2f}%"
    }]

    res_data = [{
        'Model': 'Baseline (Logistic Regression)',
        'Training Time': f"{total_time:.1f} s",
        'Peak RAM': f"{peak_ram:.2f} GB",
        'Inference Latency': f"{np.mean(lr_latencies):.4f} ms/sample"
    }]

    print("\n📊 BẢNG 1: CHỈ SỐ ĐÁNH GIÁ (METRICS)")
    display(pd.DataFrame(perf_data))

    print("\n⚙️ BẢNG 2: TIÊU HAO TÀI NGUYÊN (COMPUTATIONAL COST)")
    display(pd.DataFrame(res_data))
    print(f"📌 Thông số tập dữ liệu: {n_rows:,} dòng | {X.shape[1]} đặc trưng (sau encoding) | Dung lượng trên RAM: {dataset_ram_mb:.1f} MB")


In [3]:
evaluate_baseline_logistic('../../data/raw/application_train.csv')

📂 TEST BASELINE LOGISTIC REGRESSION (RAW DATA): ../../data/raw/application_train.csv
⚡ Đang chạy 5-Fold Cross Validation cho Baseline...
   ▫️ Fold 1 | LR Val: 0.74037 (Gap: +0.00958)
   ▫️ Fold 2 | LR Val: 0.75076 (Gap: -0.00353)
   ▫️ Fold 3 | LR Val: 0.74624 (Gap: +0.00235)
   ▫️ Fold 4 | LR Val: 0.74974 (Gap: -0.00209)
   ▫️ Fold 5 | LR Val: 0.74031 (Gap: +0.00976)

🏆 KẾT QUẢ ROC-AUC BASELINE LOGISTIC REGRESSION
🔹 Logistic Regression : 0.74546

📊 BẢNG 1: CHỈ SỐ ĐÁNH GIÁ (METRICS)


,Model,ROC-AUC,Fold AUC (Mean ± Std),Train AUC,Gap (Mean ± Std),Gini (%),KS (%)
0,Baseline (Logistic Regression),0.74546,0.74548 ± 0.00446,0.74870,0.00321 ± 0.00562,49.09%,36.47%



⚙️ BẢNG 2: TIÊU HAO TÀI NGUYÊN (COMPUTATIONAL COST)


,Model,Training Time,Peak RAM,Inference Latency
0,Baseline (Logistic Regression),33.2 s,1.70 GB,0.0016 ms/sample


📌 Thông số tập dữ liệu: 307,511 dòng | 228 đặc trưng (sau encoding) | Dung lượng trên RAM: 325.2 MB
